In [1]:
import polars as pl
import pandas as pd
import numpy as np
import pickle as pkl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import catboost as cb
from processor import PolarsLoader, ExprProcessor, PandasConverter
from IPython.display import Markdown

In [3]:
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

skf = StratifiedKFold(3, random_state = 123, shuffle = True)
ss = StratifiedShuffleSplit(1, train_size = 0.8, random_state = 123)
ss_v = StratifiedShuffleSplit(1, train_size = 0.9, random_state = 123)

In [4]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')
with open('grade_subgrade.pkl', 'rb') as f:
    c_map = pkl.load(f)
df_train['grade_subgrade_no'] = df_train['grade_subgrade'].map(c_map).astype('int')
df_test['grade_subgrade_no'] = df_test['grade_subgrade'].map(c_map).astype('int')
df_train.shape, df_test.shape

((593994, 13), (254569, 12))

In [5]:
X_all = df_test.columns.tolist()
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade_no']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [6]:
import importlib
from modeler import Experimenter

In [7]:
e = Experimenter(df_train.sample(frac = 0.01, random_state = 123), 'exp', sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [8]:
Markdown(
    e.desc_spec()
)

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=3, random_state=123, shuffle=True)` |
| **Inner Splitter (sp_v)** | `StratifiedShuffleSplit(n_splits=1, random_state=123)` |
| **Splitter Params** | `{y='loan_paid_back'}` |
| **Outer Folds** | 3 |
| **Inner Folds** | 1 |

In [9]:
e.add_grp('clf', 'exp', parent_grp = None, edges = [(None, [y])], y = y, method = 'predict_proba')
e.add_grp('preprocessor', 'pipe', parent_grp = None, method = 'transform')

In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])
e.set_node('ohe', 'preprocessor', OneHotEncoder, edges = [(None, X_cat)], params={'sparse_output': False})
e.build()

🔄 Building 2 node(s)
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
🔄 Building 2 node(s)
✅ Build complete!


In [11]:
e.rename_grp('preprocessor', 'preproc')

In [12]:
e.rename_grp('preproc', 'preprocessor')

In [13]:
e.build()

🔄 Building 0 node(s)
🔄 Building 0 node(s)
✅ Build complete!


In [14]:
e.build(rebuild=True)

🔄 Building 2 node(s)
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
🔄 Building 2 node(s)
✅ Build complete!


In [15]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression)

In [16]:
from modeler import col
e.set_node('lr1', 'lr', edges = [('std', None)])
e.set_node('lr2', 'lr', edges = [('std', None), ('ohe', col.ohe_drop_first)])

In [17]:
e.build()

🔄 Building 0 node(s)
🔄 Building 0 node(s)
✅ Build complete!


In [18]:
results = e.nodes['lr1'].experiment(0, ['output'])
results

<generator object Node.experiment at 0x7f7f91db1d00>

In [19]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [20]:
Markdown(
    e.desc_node('lr2', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr2["lr2"]
        lr2_dummy[ ]
        style lr2_dummy fill:none,stroke:none
    end
    style node_lr2 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr2
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr2
    node_std --> node_lr2
```

**Path from Root to 'lr2' (3 path(s) found)**

In [21]:
e.add_grp('dim_reduction', parent_grp = 'preprocessor')

In [22]:
e.nodes['std'].output_edges

['lr1', 'lr2']

In [23]:
from sklearn.decomposition import PCA
e.set_node('pca', 'dim_reduction', processor=PCA, edges = [('std', None)], params={'n_components': 0.9})

In [24]:
e.nodes['std'].output_edges, e.nodes['pca'].grp.role, e.nodes['pca'].status

(['lr1', 'lr2', 'pca'], 'pipe', None)

In [25]:
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])
e.build()

  └─ Effeced 3 dependent node(s): ['lr1', 'lr2', 'pca']
🔄 Building 2 node(s)
  ├─ Building 'std'...
  ├─ Building 'pca'...
  ├─ Building 'std'...
  ├─ Building 'pca'...
  ├─ Building 'std'...
  ├─ Building 'pca'...
🔄 Building 2 node(s)
✅ Build complete!


In [26]:
e.exp('lr*', retry=True)

🔄 Experimenting 2 node(s)
0 fold
  ├─ Experimenting 'lr2'...
  ├─ Experimenting 'lr1'...
1 fold
  ├─ Experimenting 'lr2'...
  ├─ Experimenting 'lr1'...
2 fold
  ├─ Experimenting 'lr2'...
  ├─ Experimenting 'lr1'...
✅ Experimentation complete!


In [27]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        subgraph grp_dim_reduction["dim_reduction"]
            grp_dim_reduction_count["1 node(s)"]
            style grp_dim_reduction_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [28]:
e.set_node('lr3', 'lr', edges = [('ohe', None), ('pca', None)])

In [29]:
Markdown(
    e.desc_node('lr3', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_dummy[ ]
        style lr3_dummy fill:none,stroke:none
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_dummy[ ]
        style pca_dummy fill:none,stroke:none
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (3 path(s) found)**

In [30]:
Markdown(
    e.desc_node('lr3', direction = 'LR', show_params=True)
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>LogisticRegression</td></tr><tr><td align='left'><b>method</b></td><td align='left'>predict_proba</td></tr>"]
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>OneHotEncoder</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>sparse_output</b></td><td align='left'>False</td></tr></table>"]
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>PCA</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>n_components</b></td><td align='left'>0.9</td></tr></table>"]
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>StandardScaler</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr>"]
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (3 path(s) found)**

In [31]:
e.add_grp('cb', parent_grp='clf', processor=cb.CatBoostClassifier, params={'verbose': 0})

In [32]:
e.set_node('cb1',  grp = 'cb', edges = [(None, X_num), (None, X_cat)], params = {'cat_features': X_cat})

In [33]:
import lightgbm as lgb

In [34]:
e.add_grp('lgb', parent_grp='clf', processor=lgb.LGBMClassifier, params={'verbose': -1})

In [35]:
e.set_node('lgb1',  grp = 'lgb', edges = [(None, X_num), (None, X_cat)], params={'categorical_features': X_cat})

In [36]:
from modeler._metric import Metric
from modeler._stacking import Stacking
e.add_metric('AUC', [(None, y)], slice(-1, None), roc_auc_score, include_train = True)
e.add_metric('AUC2', [(None, y)], slice(None, 1), roc_auc_score, include_train = True)
e.add_stacking('S1',[(None, y)], slice(-1, None))
e.add_stacking('S2', [(None, y)], slice(None, 1))

In [37]:
e.exp(None)

🔄 Experimenting 3 node(s)
0 fold
  ├─ Experimenting 'lgb1'...
  Progress: 100/100 (100.0%) | training-binary_logloss: 0.0820, valid_1-binary_logloss: 0.2943  ├─ Experimenting 'cb1'...


/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


  ├─ Experimenting 'lr3'...
1 fold
  ├─ Experimenting 'lgb1'...
  Progress: 100/100 (100.0%) | training-binary_logloss: 0.0752, valid_1-binary_logloss: 0.2759  ├─ Experimenting 'cb1'...


/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


  ├─ Experimenting 'lr3'...
2 fold
  ├─ Experimenting 'lgb1'...
  Progress: 100/100 (100.0%) | training-binary_logloss: 0.0738, valid_1-binary_logloss: 0.2509  ├─ Experimenting 'cb1'...


/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


  ├─ Experimenting 'lr3'...
✅ Experimentation complete!


In [38]:
e.exp(None)

🔄 Experimenting 0 node(s)
0 fold
1 fold
2 fold
✅ Experimentation complete!


In [39]:
e.nodes['lr1'].status

'built'

In [40]:
e.stacking['S1'].get_dataset(None)

,lr4__loan_paid_back_1,lr3__loan_paid_back_1,lgb1__loan_paid_back_1,cb1__loan_paid_back_1,loan_paid_back
id,,,,,
176836,0.715060,0.712976,0.679160,0.526876,1
321322,0.913445,0.949090,0.969608,0.944185,0
225907,0.781299,0.744901,0.396050,0.540851,0
290104,0.694982,0.689015,0.661095,0.514444,1
453658,0.941816,0.932390,0.951571,0.934432,1
...,...,...,...,...,...
425863,0.966170,0.961187,0.993941,0.987203,1
103194,0.977573,0.960983,0.996805,0.971427,1
252332,0.010544,0.008886,0.005673,0.008119,0


In [41]:
e.stacking['S1']._get_nodes(None)

['lr4', 'lr3', 'lgb1', 'cb1']

In [42]:
e.stacking['S2'].get_dataset(None)

,lr4__loan_paid_back_0,lr3__loan_paid_back_0,lgb1__loan_paid_back_0,cb1__loan_paid_back_0,loan_paid_back
id,,,,,
176836,0.284940,0.287024,0.320840,0.473124,1
321322,0.086555,0.050910,0.030392,0.055815,0
225907,0.218701,0.255099,0.603950,0.459149,0
290104,0.305018,0.310985,0.338905,0.485556,1
453658,0.058184,0.067610,0.048429,0.065568,1
...,...,...,...,...,...
425863,0.033830,0.038813,0.006059,0.012797,1
103194,0.022427,0.039017,0.003195,0.028573,1
252332,0.989456,0.991114,0.994327,0.991881,0


In [43]:
e.metric['AUC2'].get_metrics(None)

0                             1                             2  \
             0                             0                             0   
         valid train_sub valid_sub     valid train_sub valid_sub     valid   
lgb1  0.084511  0.000489  0.116080  0.101826  0.000114  0.093280  0.105895   
cb1   0.074635  0.058834  0.088049  0.085150  0.061278  0.061374  0.092149   
lr3   0.077149  0.084252  0.096674  0.087862  0.083204  0.056423  0.093097   

                          
                          
     train_sub valid_sub  
lgb1  0.000415  0.082498  
cb1   0.033871  0.085733  
lr3   0.077231  0.094198

In [44]:
e.nodes['lr1'].status

'built'

In [45]:
e.finalize('lr1')

  ├─ Finalize 'lr1'


In [46]:
e.nodes['lr1'].status

'finalized'

In [47]:
del e

In [48]:
e = Experimenter.load('exp', df_train.sample(frac = 0.01, random_state = 123))

📂 Loading Experimenter from exp...
   Loading 6 group(s)...
   Loading 8 node(s)...
   Loading 2 metric(s)...
   Loading 2 stacking(s)...
✅ Experimenter loaded successfully
   - 8 node(s)
   - 6 group(s)
   - 3 fold(s)


In [49]:
e.metric

{'AUC': <modeler._metric.Metric at 0x7f7f083bf5f0>,
 'AUC2': <modeler._metric.Metric at 0x7f7f042470b0>}

In [50]:
e.stacking['S1']._get_nodes(None)

['lr4', 'lr3', 'lgb1', 'cb1']

In [51]:
e.nodes['lr1'].status

'finalized'

In [52]:
e.desc_node_vars('lr3', 0)

(                                          name
 node seq                                      
 ohe  0                      ohe__gender_Female
      1                        ohe__gender_Male
      2                       ohe__gender_Other
      3            ohe__marital_status_Divorced
      4             ohe__marital_status_Married
      5              ohe__marital_status_Single
      6             ohe__marital_status_Widowed
      7         ohe__education_level_Bachelor's
      8        ohe__education_level_High School
      9           ohe__education_level_Master's
      10             ohe__education_level_Other
      11               ohe__education_level_PhD
      12        ohe__employment_status_Employed
      13         ohe__employment_status_Retired
      14   ohe__employment_status_Self-employed
      15         ohe__employment_status_Student
      16      ohe__employment_status_Unemployed
      17             ohe__loan_purpose_Business
      18                  ohe__loan_purp

In [53]:
Markdown(
    e.desc_node('cb1', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_cb1["cb1"]
        cb1_dummy[ ]
        style cb1_dummy fill:none,stroke:none
    end
    style node_cb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    Root --> node_cb1
```

**Path from Root to 'cb1' (1 path(s) found)**

In [54]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        subgraph grp_dim_reduction["dim_reduction"]
            grp_dim_reduction_count["1 node(s)"]
            style grp_dim_reduction_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["3 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["1 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lgb["lgb"]
            grp_lgb_count["1 node(s)"]
            style grp_lgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [55]:
Markdown(
    e.desc_node('lgb1', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lgb1["lgb1"]
        lgb1_dummy[ ]
        style lgb1_dummy fill:none,stroke:none
    end
    style node_lgb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    Root --> node_lgb1
```

**Path from Root to 'lgb1' (1 path(s) found)**

In [56]:
e._find_descendants('std')

{'lr1', 'lr2', 'lr3', 'pca'}

In [57]:
# e = Experimenter(df_train, sp = skf, sp_v = None, splitter_params = {'y': y})

In [58]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        subgraph grp_dim_reduction["dim_reduction"]
            grp_dim_reduction_count["1 node(s)"]
            style grp_dim_reduction_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["3 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["1 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lgb["lgb"]
            grp_lgb_count["1 node(s)"]
            style grp_lgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [59]:
from sklearn.preprocessing import TargetEncoder

e.set_node('tgt', 'preprocessor', TargetEncoder, edges = [(None, X_cat), (None, y)], y = y, params={'target_type': 'binary'}, method = 'fit_transform')

In [60]:
e.build()

🔄 Building 1 node(s)
  ├─ Building 'tgt'...
  ├─ Building 'tgt'...
  ├─ Building 'tgt'...
🔄 Building 1 node(s)
✅ Build complete!


In [61]:
e.set_node('lr4', 'lr', LogisticRegression, edges =  [('std', None), ('ohe', None), ('tgt', None), ('pca', None)])

In [64]:
e.exp(None)

🔄 Experimenting 1 node(s)
0 fold
  ├─ Experimenting 'lr4'...
1 fold
  ├─ Experimenting 'lr4'...
2 fold
  ├─ Experimenting 'lr4'...
✅ Experimentation complete!


In [65]:
e.desc_node_vars('lr4', 0)

(                                          name
 node seq                                      
 std  0                      std__annual_income
      1               std__debt_to_income_ratio
      2                       std__credit_score
      3                        std__loan_amount
      4                      std__interest_rate
      5                  std__grade_subgrade_no
 ohe  0                      ohe__gender_Female
      1                        ohe__gender_Male
      2                       ohe__gender_Other
      3            ohe__marital_status_Divorced
      4             ohe__marital_status_Married
      5              ohe__marital_status_Single
      6             ohe__marital_status_Widowed
      7         ohe__education_level_Bachelor's
      8        ohe__education_level_High School
      9           ohe__education_level_Master's
      10             ohe__education_level_Other
      11               ohe__education_level_PhD
      12        ohe__employment_status_E

In [67]:
e.metric['AUC'].get_metrics(None)

0                             1                             2  \
             0                             0                             0   
         valid train_sub valid_sub     valid train_sub valid_sub     valid   
lgb1  0.915489  0.999511  0.883920  0.898174  0.999886  0.906720  0.894105   
cb1   0.925365  0.941166  0.911951  0.914850  0.938722  0.938626  0.907851   
lr3   0.922851  0.915748  0.903326  0.912138  0.916796  0.943577  0.906903   
lr4   0.923303  0.917678  0.904804  0.913129  0.918571  0.943817  0.910715   

                          
                          
     train_sub valid_sub  
lgb1  0.999585  0.917502  
cb1   0.966129  0.914267  
lr3   0.922769  0.905802  
lr4   0.923713  0.899213

In [68]:
from modeler import create_like

In [64]:
e2 = create_like(e, df_train.sample(frac = 0.01, random_state = 123), sp = ss, sp_v = StratifiedKFold(2, random_state=123, shuffle = True),
                splitter_params = {'y': y})

🔄 Creating new Experimenter with same structure...


TypeError: Experimenter.__init__() got an unexpected keyword argument 'stacking'

In [ ]:
e2.build()

In [ ]:
lr_a.get_coef('lr1').T.groupby(level = [2]).mean().T

In [ ]:
lr_a.get_intercept('lr1').T.groupby(level = [2]).mean().T

In [ ]:
e.root.data

In [ ]:
for i in e2.get_node_valid_output(0, 'cb1', slice(0, -1)):
    print(i.data)

In [ ]:
e2.get_data_valid(0, [('lr1', slice(0, -1))])

In [ ]:
for i in e2.get_data_valid(0, [('lr1', slice(0, -1))]):
    print(i.data)

In [ ]:
e3 = create_like(e, df_train.sample(frac = 0.1, random_state = 123), splitter_params = {'y': y})

In [ ]:
e3.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [ ]:
e3.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')

In [ ]:
e3.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [ ]:
e3.set_node('lr1', 'lr')

In [ ]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    #sgpp.PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')

In [ ]:
X_all = df_test.columns
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [ ]:
e = Experimenter(df_train, sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [ ]:
e.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')
e.add_grp('preprocessor', method = 'transform')

In [ ]:
from sklearn.preprocessing import StandardScaler
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [ ]:
for a, b in e.get_data(0, [(None, X_num)]):
    print(a[0].data, b)

In [ ]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [ ]:
e.set_node('lr1', 'lr')

In [ ]:
for i in e.get_data_valid(0, [(None, y)]):
    print(i.data)

In [ ]:
ss = StratifiedKFold(n_splits=3)
for a, b in ss.split(df_train[X_all],  df_train[y]):
    pass

In [ ]:
lr = LogisticRegression()
lr.fit(df_train[X_num], df_train[[y]])

In [ ]:
df_train.to_pandas()[[y]].shape

In [ ]:
e.nodes['lr1'].y